# Reviewer revision: outer results, figures and tables
Read-only analysis of the completed expanded run. This notebook replaces the historical figure notebook, including its embedded outputs. No model is fitted, threshold retuned, result file overwritten or paper edited here. Use Revision-Evaluation.ipynb or SERVER_RUN.md to run the canonical pipeline first.

Primary discrimination uses the prespecified auc_selected stream. Sensitivity-oriented referral uses sensitivity_selected. Do not choose a reporting stream from outer performance.


In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_revision.py").is_file()
)
sys.path.insert(0, str(ROOT))


def read_json(path):
    return json.loads(Path(path).read_text())


def table(path, **kwargs):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"**Unavailable:** {path.relative_to(ROOT)}"))
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except pd.errors.EmptyDataError:
        display(Markdown(f"**No estimable rows:** {path.relative_to(ROOT)}"))
        return pd.DataFrame()


RESULTS = ROOT / os.environ.get("IIVV_RESULTS", "results/revision-expanded")
manifest = read_json(RESULTS / "run_manifest.json")
if manifest.get("status") != "completed" or manifest.get("synthetic"):
    raise ValueError("A completed, non-synthetic clinical run is required.")
cfg = manifest["config"]
if (cfg["outer_folds"], cfg["inner_folds"], cfg["outer_repeats"]) != (5, 4, 10):
    raise ValueError("This notebook is for the prespecified expanded protocol.")
from revision.verification import verify_results

display(verify_results(RESULTS, verify_inputs=True))
TASKS = cfg["tasks"]
metrics = table(RESULTS / "all_metrics.csv")
metrics = metrics[metrics.evaluation == "nested_cv"]
primary = metrics[
    metrics.stream.isin(
        [
            "auc_selected",
            "sensitivity_selected",
            "baseline_svm",
            "baseline_rf",
            "baseline_lr",
        ]
    )
]
display(cfg)
display(read_json(RESULTS / "analysis_status.json"))

## 1. Leakage boundaries, fold demographics and labels — R1 C1–3, C7–8, C10, M6–8; R3 C1–2
Split files preserve patient identity and all inner/outer partitions. Show counts by pathology, anatomy and quality cutoff. Unique-patient counts across pathology rows may overlap.


In [ ]:
for task in TASKS:
    display(Markdown(f"### {task}"))
    display(read_json(RESULTS / task / "classes.json"))
    counts = table(RESULTS / task / "split_population_counts.csv")
    display(
        counts[
            (counts.location == "All locations") & (counts.partition == "outer_test")
        ]
    )
    display(
        table(RESULTS / task / "task_inclusion.csv", dtype={"patient_id": str})
        .groupby("pathology")
        .agg(n_sites=("sample_id", "size"), n_patients=("patient_id", "nunique"))
    )

## 2. Prespecified baseline and selected pipelines — R1 C10, C15–16
Compare site, equal-patient-weighted site and explicitly defined patient endpoints separately. Read covered-case metrics together with coverage; abstention is not a negative diagnosis. Macro measures take priority over micro summaries.


In [ ]:
columns = [
    "task",
    "stream",
    "unit",
    "n_sites",
    "n_covered",
    "n_patients",
    "coverage",
    "macro_auc",
    "macro_f1",
    "balanced_accuracy",
    "accuracy",
    "brier",
    "log_loss",
]
display(primary[columns].sort_values(["task", "unit", "stream"]))
for task in TASKS:
    display(Markdown(f"### {task}: patient endpoint"))
    display(read_json(RESULTS / task / "nested_cv_auc_selected_patient_endpoint.json"))
    display(
        table(RESULTS / task / "nested_cv_auc_selected_site_confidence_intervals.csv")
    )

## 3. All class metrics and dysplasia error destinations — R1 C15–16
TP/FN and false-negative destinations refer to covered sites. Class-specific patient-cluster intervals quantify conditional uncertainty. Dysplasia predicted as cancer is an exact-class error even if both belong to the combined referral-positive class.


In [ ]:
perclass = pd.concat(
    [table(RESULTS / task / "per_class_metrics.csv") for task in TASKS],
    ignore_index=True,
)
selected_classes = perclass[
    (perclass.evaluation == "nested_cv")
    & (perclass.unit == "site")
    & (perclass.stream == "auc_selected")
]
display(selected_classes)
for task in ["cancer_vs_lr_hr", "MC"]:
    display(Markdown(f"### {task}: high-grade dysplasia"))
    cm = table(
        RESULTS / task / "nested_cv_auc_selected_site_confusion_counts.csv", index_col=0
    )
    display(cm)
    display(cm.loc[["High-grade dysplasia"]])
    ci = table(
        RESULTS
        / task
        / "nested_cv_auc_selected_site_per_class_confidence_intervals.csv"
    )
    display(ci[ci.class_name == "High-grade dysplasia"])

## 4. Quality thresholds, supervised selectors and complex features — R1 C2–6, C9, C18; R3 C3–4
These are fixed-reference pipeline ablations. Supervised rankings summarize each training site's spectrum; PCA uses training site/pattern rows. Complex pipelines rank their first component; circuit parameters still require the full raw spectrum. Interpret coverage differences and unavailable comparisons. Do not select a new primary pipeline here.


In [ ]:
ablations = metrics[
    (metrics.unit == "site") & metrics.stream.str.startswith("ablation_")
]
display(
    ablations[
        ["task", "stream", "macro_auc", "macro_f1", "balanced_accuracy", "coverage"]
    ].sort_values(["task", "stream"])
)
display(table(RESULTS / "quality/retention_by_threshold_class.csv"))
display(table(RESULTS / "quality/missing_patterns_by_class.csv"))
display(table(RESULTS / "quality/quality_cluster_associations.csv"))
for task in TASKS:
    display(Markdown(f"### {task}: incomplete ablations"))
    display(read_json(RESULTS / task / "unavailable_streams.json"))

## 5. Available confounding and anatomy — R1 C5, C12–14
Compare clinical-only, matched EIS-only LR and EIS-plus-clinical LR on identical patient folds. Identical EIS rules and LR settings separate covariate addition from classifier changes. These are descriptive controls; they cannot adjust unrecorded variables or establish causality.


In [ ]:
cov = metrics[
    (metrics.unit == "site")
    & metrics.stream.isin(
        ["covariate_only", "covariate_eis_only", "covariate_combined"]
    )
]
display(cov[["task", "stream", "macro_auc", "macro_f1", "coverage"]])
display(read_json(RESULTS / "quality/clinical_metadata_status.json"))
display(table(RESULTS / "quality/class_by_location.csv"))
for task in TASKS:
    location = table(RESULTS / task / "location_stratified_metrics.csv")
    if not location.empty:
        display(location[location.stream == "auc_selected"])

## 6. Referral, operating points and prevalence — R1 C30; R2 C3–5
Training sensitivity target 0.95 is not a guaranteed test sensitivity or a validated safety threshold. Specificity targets are also training-selected. Report sensitivity, specificity, PPV, NPV, coverage and referral fraction. The fallback refers unevaluable sites; prevalence projections assume transportable sensitivity/specificity.


In [ ]:
binary_tasks = ["healthy_vs_cancer", "highrisk_vs_lowrisk"]
for task in binary_tasks:
    display(Markdown(f"### {task}"))
    display(
        primary[
            (primary.task == task)
            & (primary.stream == "sensitivity_selected")
            & (primary.unit == "site")
        ]
    )
    prefix = RESULTS / task / "nested_cv_sensitivity_selected"
    display(table(Path(str(prefix) + "_operating_points.csv")))
    display(read_json(Path(str(prefix) + "_referral_workflow.json")))
    projection = table(Path(str(prefix) + "_prevalence_projections.csv"))
    display(projection[projection.class_index == 1])

## 7. Calibration and decision curves — R1 C30
Curves use saved outer predictions at study prevalence. Net benefit is exploratory in this surgical cohort. Treat-all/treat-none references are essential; these curves do not demonstrate safe biopsy avoidance.


In [ ]:
import matplotlib.pyplot as plt

for task in binary_tasks:
    positive = read_json(RESULTS / task / "classes.json")["positive_class"]
    prefix = RESULTS / task / "nested_cv_sensitivity_selected"
    calibration = table(Path(str(prefix) + "_calibration.csv"))
    decision = table(Path(str(prefix) + "_decision_curve.csv"))
    calibration = calibration[calibration.class_name == positive]
    decision = decision[decision.class_name == positive]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot([0, 1], [0, 1], "k--")
    axes[0].plot(calibration.mean_probability, calibration.observed_fraction, "o-")
    axes[0].set(
        xlabel="Predicted probability",
        ylabel="Observed fraction",
        title=f"{task}: {positive}",
    )
    axes[1].plot(decision.threshold, decision.net_benefit, label="Model")
    axes[1].plot(decision.threshold, decision.treat_all, "--", label="Refer all")
    axes[1].plot(decision.threshold, decision.treat_none, "k:", label="Refer none")
    axes[1].set(
        xlabel="Referral threshold",
        ylabel="Net benefit (study prevalence)",
        ylim=(-0.25, 1),
    )
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 8. Patient-cluster uncertainty and multiplicity — R1 C17–19
Intervals and block permutation tests condition on fitted outer predictions. They do not account for full refitting uncertainty. Repeated predictions are aggregated once per site; folds/repeats are not independent patients. All planned task/model/objective contrasts remain in the Holm family, including unavailable tests.


In [ ]:
paired = table(RESULTS / "paired_comparisons_holm.csv")
assert len(paired) == len(TASKS) * len(cfg["models"]) * len(cfg["objectives"]) == 48
display(paired.sort_values(["task", "model", "objective"]))
display(
    {
        "planned_comparisons": len(paired),
        "unavailable_p_values": int(paired.p_value.isna().sum()),
        "holm_below_0_05": int(
            (paired.p_holm_all_tasks_models_objectives < 0.05).sum()
        ),
    }
)

## 9. Frozen frequency/pattern identities, stability and resources — R1 C20–21, C26–27; R3 C3–4
Use actual outer-training coordinate identities, not ranked subset sizes. Stability is descriptive across overlapping training sets. Circuit inputs and solver dimensions are distinct from acquired spectra. Software batch timings do not measure acquisition latency; electrode/drive counts do not validate a simplified device.


In [ ]:
for task in TASKS:
    display(Markdown(f"### {task}"))
    identities = read_json(RESULTS / task / "frequency_and_pattern_identities.json")
    display(
        pd.DataFrame(
            [r for r in identities["selections"] if r["stream"] == "auc_selected"]
        )
    )
    for filename in [
        "ranking_stability.csv",
        "hardware_requirements.csv",
        "measured_computation_times.csv",
    ]:
        values = table(RESULTS / task / filename)
        if not values.empty:
            display(values[values.stream == "auc_selected"])

## 10. Repeatability, provenance and unresolved evidence — R1 C11, C24–25; R2 C1–2
Use test-mean CV/ICC endpoints, not an assumed nine-burst array. See mat-to-npz.ipynb for source availability, pathology/contact evidence gaps and metadata audits. Retain failed-candidate records even when the run completed.


In [ ]:
display(read_json(RESULTS / "repeatability/status.json"))
display(table(RESULTS / "repeatability/icc_and_inter_site_variability.csv"))
display(
    {
        "source_versions": manifest.get("source_hashes"),
        "versions": manifest.get("versions"),
        "resume_history": manifest.get("resume_history", []),
    }
)
for task in TASKS:
    failures = read_json(RESULTS / task / "candidate_failures.json")
    display(Markdown(f"### {task}: {len(failures)} failed candidates"))
    display(pd.DataFrame(failures).head(20))

## 11. Regenerated confusion and ROC figures
Display only the current run's saved figures. Historical repository plots remain separate.


In [ ]:
from IPython.display import Image

for task in TASKS:
    folder = RESULTS / f"plots_{task}_auc" / "auc_selected"
    display(Markdown(f"### {task}"))
    for name in ["CM.png", "CM_norm.png", "ROC.png"]:
        path = folder / name
        if path.is_file():
            display(Image(filename=str(path), width=650))
        else:
            display(Markdown(f"**Figure unavailable:** {name}"))

## Handoff
Return the complete run tree and logs for review. Manuscript/rebuttal imports are a later step, using the guarded scripts listed in SERVER_RUN.md. No cell in this notebook changes manuscript sources or writes new estimates.
